# 03 · Feature Engineering (Titanic) — deep dive

> **Course:** Data Analytics Laboratory · IIIT Dharwad · companion to [`../03-Feature-Engineering.md`](../03-Feature-Engineering.md)
>
> Every number quoted in the note is produced here. Run top to bottom from this folder (needs `titanic.csv`, the Kaggle `train.csv`, next to it). Tested on pandas 3.0, scikit-learn 1.9.

| Part | Topic |
|:-:|:--|
| A | Load & inspect; who survived? |
| B | The manual's code, run as written (pandas 3 pitfalls) |
| C | Missing values: global vs group-wise imputation, missingness as a feature |
| D | Creating new features: FamilySize, IsAlone, Title, AgeGroup, HasCabin |
| E | Encoding: LabelEncoder, OrdinalEncoder, one-hot |
| F | Scaling: Standard, MinMax, Robust; log transform |
| G | Data leakage and the fix: fit on train only, `Pipeline` + `ColumnTransformer` |
| H | Does feature engineering help? Baseline vs engineered, LR vs RF, cross-validation |
| I | Practice-problem solutions |

In [1]:
import numpy as np
import pandas as pd
import sklearn
import warnings

from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score

pd.set_option("display.width", 130)
print("pandas", pd.__version__, "| numpy", np.__version__, "| scikit-learn", sklearn.__version__)

pandas 3.0.3 | numpy 2.4.6 | scikit-learn 1.9.0


## Part A — Load & inspect; who survived?

In [2]:
raw = pd.read_csv("titanic.csv")
print(raw.shape)
print(raw.isna().sum()[raw.isna().sum() > 0])
print("\nOverall survival rate:", round(raw["Survived"].mean(), 4))

(891, 12)
Age         177
Cabin       687
Embarked      2
dtype: int64

Overall survival rate: 0.3838


In [3]:
# survival rate by the raw categorical columns
for col in ["Sex", "Pclass", "Embarked"]:
    print(raw.groupby(col)["Survived"].agg(["mean", "count"]).round(3), "\n")

         mean  count
Sex                 
female  0.742    314
male    0.189    577 

         mean  count
Pclass              
1       0.630    216
2       0.473    184
3       0.242    491 

           mean  count
Embarked              
C         0.554    168
Q         0.390     77
S         0.337    644 



In [4]:
print(pd.crosstab([raw["Sex"]], raw["Pclass"], values=raw["Survived"], aggfunc="mean").round(3))
print("\nFare skewness:", round(raw["Fare"].skew(), 3), "| Age skewness:", round(raw["Age"].skew(), 3))
print(raw[["Age", "Fare"]].describe().round(2))

Pclass      1      2      3
Sex                        
female  0.968  0.921  0.500
male    0.369  0.157  0.135

Fare skewness: 4.787 | Age skewness: 0.389
          Age    Fare
count  714.00  891.00
mean    29.70   32.20
std     14.53   49.69
min      0.42    0.00
25%     20.12    7.91
50%     28.00   14.45
75%     38.00   31.00
max     80.00  512.33


## Part B — The manual's code, run as written

The manual fills missing values with `train_df['Age'].fillna(..., inplace=True)`. In **pandas 3** (Copy-on-Write), `train_df['Age']` is a new object, so the in-place fill changes that temporary object and **not** `train_df`.

In [5]:
m = raw.copy()
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    m["Age"].fillna(m["Age"].median(), inplace=True)
    print("Warnings raised:", [type(x.message).__name__ for x in w])
print("Missing Age after the manual's inplace fill:", m["Age"].isna().sum())

m["Age"] = m["Age"].fillna(m["Age"].median())          # the fix: assign back
print("Missing Age after assigning back:          ", m["Age"].isna().sum())

Warnings raised: ['ChainedAssignmentError']
Missing Age after the manual's inplace fill: 177
Missing Age after assigning back:           0


In [6]:
# Fare has no missing values in train.csv (only test.csv has one), so the manual's Fare fill is a no-op here
print("Missing Fare in train.csv:", raw["Fare"].isna().sum())

Missing Fare in train.csv: 0


### The manual's whole pipeline, with only the pandas-3 fix applied
Steps exactly as in Experiment 4: fill → drop Cabin → FamilySize, IsAlone, Title, AgeGroup → LabelEncode Title and AgeGroup → one-hot Sex, Embarked, AgeGroup → StandardScaler on Age, Fare, FamilySize → drop Name, Ticket, PassengerId → split 80:20 → RandomForest.

In [7]:
def manual_pipeline(df):
    df = df.copy()
    df["Age"] = df["Age"].fillna(df["Age"].median())
    df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
    df["Fare"] = df["Fare"].fillna(df["Fare"].median())
    df = df.drop(columns=["Cabin"])
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    df["Title"] = df["Name"].apply(lambda name: name.split(",")[1].split(".")[0].strip())
    df["AgeGroup"] = pd.cut(df["Age"], bins=[0, 12, 18, 30, 50, 100],
                            labels=["Child", "Teen", "YoungAdult", "Adult", "Senior"])
    le = LabelEncoder()
    df["Title"] = le.fit_transform(df["Title"])
    df["AgeGroup"] = le.fit_transform(df["AgeGroup"])
    df = pd.get_dummies(df, columns=["Sex", "Embarked", "AgeGroup"])
    sc = StandardScaler()
    df[["Age", "Fare", "FamilySize"]] = sc.fit_transform(df[["Age", "Fare", "FamilySize"]])
    df = df.drop(columns=["Name", "Ticket", "PassengerId"])
    return df

mdf = manual_pipeline(raw)
print(mdf.shape)
print(list(mdf.columns))
X = mdf.drop(columns=["Survived"]); y = mdf["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
print(f"Manual pipeline, RandomForest accuracy: {accuracy_score(y_test, rf.predict(X_test)):.4f}")

(891, 19)
['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'Title', 'Sex_female', 'Sex_male', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'AgeGroup_0', 'AgeGroup_1', 'AgeGroup_2', 'AgeGroup_3', 'AgeGroup_4']


Manual pipeline, RandomForest accuracy: 0.8045


In [8]:
# What LabelEncoder did to AgeGroup: alphabetical codes, not the age order
ag = pd.cut(raw["Age"].fillna(raw["Age"].median()), bins=[0, 12, 18, 30, 50, 100],
            labels=["Child", "Teen", "YoungAdult", "Adult", "Senior"])
le = LabelEncoder().fit(ag)
print("LabelEncoder classes (code order):", list(le.classes_))
print("Natural order                     :", list(ag.cat.categories))
print("Category codes (keep the order)   :", dict(zip(ag.cat.categories, range(5))))

LabelEncoder classes (code order): ['Adult', 'Child', 'Senior', 'Teen', 'YoungAdult']
Natural order                     : ['Child', 'Teen', 'YoungAdult', 'Adult', 'Senior']
Category codes (keep the order)   : {'Child': 0, 'Teen': 1, 'YoungAdult': 2, 'Adult': 3, 'Senior': 4}


## Part C — Missing values

`Age` is missing for 177 passengers. Filling them all with one global median (28) piles 177 identical values at 28. A better estimate uses another column that predicts age: the **title** (`Master` = boy, `Miss` = often young, `Mrs` = married woman).

In [9]:
df = raw.copy()
df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
print(df["Title"].value_counts())

Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64


In [10]:
print("Global Age median:", df["Age"].median(), "| mean:", round(df["Age"].mean(), 2))
print("\nMedian Age by Title (titles with missing ages):")
miss_titles = df.loc[df["Age"].isna(), "Title"].value_counts()
print(pd.DataFrame({"missing": miss_titles, "median_age": df.groupby("Title")["Age"].median()[miss_titles.index]}))

Global Age median: 28.0 | mean: 29.7

Median Age by Title (titles with missing ages):
        missing  median_age
Title                      
Mr          119        30.0
Miss         36        21.0
Mrs          17        35.0
Master        4         3.5
Dr            1        46.5


In [11]:
global_fill = df["Age"].fillna(df["Age"].median())
title_fill = df["Age"].fillna(df.groupby("Title")["Age"].transform("median"))
print("Std of Age  original (known only): {:.3f}".format(df["Age"].std()))
print("Std of Age  global-median fill   : {:.3f}".format(global_fill.std()))
print("Std of Age  title-median fill    : {:.3f}".format(title_fill.std()))
print("Rows equal to 28 after global fill:", (global_fill == 28).sum(), "| after title fill:", (title_fill == 28).sum())

Std of Age  original (known only): 14.526
Std of Age  global-median fill   : 13.020
Std of Age  title-median fill    : 13.265
Rows equal to 28 after global fill: 202 | after title fill: 25


In [12]:
# Missingness itself can carry information
df["HasCabin"] = df["Cabin"].notna().astype(int)
df["AgeMissing"] = df["Age"].isna().astype(int)
print(df.groupby("HasCabin")["Survived"].agg(["mean", "count"]).round(3))
print()
print(df.groupby("AgeMissing")["Survived"].agg(["mean", "count"]).round(3))
print()
print(pd.crosstab(df["Pclass"], df["HasCabin"], normalize="index").round(3))

           mean  count
HasCabin              
0         0.300    687
1         0.667    204

             mean  count
AgeMissing              
0           0.406    714
1           0.294    177

HasCabin      0      1
Pclass                
1         0.185  0.815
2         0.913  0.087
3         0.976  0.024


## Part D — Creating new features

In [13]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
print(df.groupby("FamilySize")["Survived"].agg(["mean", "count"]).round(3))
print()
print(df.groupby("IsAlone")["Survived"].agg(["mean", "count"]).round(3))

             mean  count
FamilySize              
1           0.304    537
2           0.553    161
3           0.578    102
4           0.724     29
5           0.200     15
6           0.136     22
7           0.333     12
8           0.000      6
11          0.000      7

          mean  count
IsAlone              
0        0.506    354
1        0.304    537


In [14]:
# family size in 3 bands
df["FamilyBand"] = pd.cut(df["FamilySize"], bins=[0, 1, 4, 11], labels=["Alone", "Small", "Large"])
print(df.groupby("FamilyBand", observed=True)["Survived"].agg(["mean", "count"]).round(3))

             mean  count
FamilyBand              
Alone       0.304    537
Small       0.579    292
Large       0.161     62


In [15]:
# Title: the manual's lambda vs a regex; group the rare titles
t_lambda = raw["Name"].apply(lambda name: name.split(",")[1].split(".")[0].strip())
print("lambda == regex for every row:", (t_lambda == df["Title"]).all())
print("Number of distinct titles:", df["Title"].nunique())

title_map = {"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"}
df["Title"] = df["Title"].replace(title_map)
common = ["Mr", "Mrs", "Miss", "Master"]
df["Title"] = df["Title"].where(df["Title"].isin(common), "Rare")
print(df.groupby("Title")["Survived"].agg(["mean", "count"]).round(3).sort_values("mean"))

lambda == regex for every row: True
Number of distinct titles: 17
         mean  count
Title               
Mr      0.157    517
Rare    0.348     23
Master  0.575     40
Miss    0.703    185
Mrs     0.794    126


In [16]:
df["Age"] = df["Age"].fillna(df.groupby("Title")["Age"].transform("median"))
df["AgeGroup"] = pd.cut(df["Age"], bins=[0, 12, 18, 30, 50, 100],
                        labels=["Child", "Teen", "YoungAdult", "Adult", "Senior"])
print(df.groupby("AgeGroup", observed=True)["Survived"].agg(["mean", "count"]).round(3))

             mean  count
AgeGroup                
Child       0.575     73
Teen        0.429     70
YoungAdult  0.311    425
Adult       0.448    259
Senior      0.344     64


In [17]:
# Two more common ideas
df["FarePerPerson"] = df["Fare"] / df["FamilySize"]
df["TicketGroup"] = df.groupby("Ticket")["Ticket"].transform("count")
print(df[["Fare", "FamilySize", "FarePerPerson", "TicketGroup"]].head(8))
print("\nPassengers sharing a ticket with someone:", (df["TicketGroup"] > 1).sum())

      Fare  FamilySize  FarePerPerson  TicketGroup
0   7.2500           2        3.62500            1
1  71.2833           2       35.64165            1
2   7.9250           1        7.92500            1
3  53.1000           2       26.55000            2
4   8.0500           1        8.05000            1
5   8.4583           1        8.45830            1
6  51.8625           1       51.86250            1
7  21.0750           5        4.21500            4

Passengers sharing a ticket with someone: 344


## Part E — Encoding categorical features

In [18]:
s = pd.Series(["S", "C", "Q", "S", "C"], name="Embarked")
print("LabelEncoder        :", LabelEncoder().fit_transform(s))
print("category codes      :", s.astype("category").cat.codes.tolist())
print("\npd.get_dummies:")
print(pd.get_dummies(s, prefix="Emb", dtype=int))
print("\npd.get_dummies(drop_first=True):")
print(pd.get_dummies(s, prefix="Emb", drop_first=True, dtype=int))

LabelEncoder        : [2 0 1 2 0]
category codes      : [2, 0, 1, 2, 0]

pd.get_dummies:
   Emb_C  Emb_Q  Emb_S
0      0      0      1
1      1      0      0
2      0      1      0
3      0      0      1
4      1      0      0

pd.get_dummies(drop_first=True):
   Emb_Q  Emb_S
0      0      1
1      0      0
2      1      0
3      0      1
4      0      0


In [19]:
# OrdinalEncoder with an explicit order (for a truly ordinal feature)
groups = pd.DataFrame({"AgeGroup": ["Teen", "Child", "Senior", "Adult", "YoungAdult"]})
oe = OrdinalEncoder(categories=[["Child", "Teen", "YoungAdult", "Adult", "Senior"]])
print(pd.concat([groups, pd.DataFrame(oe.fit_transform(groups), columns=["code"])], axis=1))

     AgeGroup  code
0        Teen   1.0
1       Child   0.0
2      Senior   4.0
3       Adult   3.0
4  YoungAdult   2.0


In [20]:
# OneHotEncoder remembers the categories it saw during fit; get_dummies does not
train_part = pd.DataFrame({"Embarked": ["S", "C", "Q", "S"]})
new_part = pd.DataFrame({"Embarked": ["S", "S"]})
print("get_dummies on new data gives columns:", list(pd.get_dummies(new_part).columns))
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(train_part)
print("OneHotEncoder columns:", list(ohe.get_feature_names_out()))
print(ohe.transform(new_part))
print("unseen category 'X' ->", ohe.transform(pd.DataFrame({"Embarked": ["X"]})))

get_dummies on new data gives columns: ['Embarked_S']
OneHotEncoder columns: ['Embarked_C', 'Embarked_Q', 'Embarked_S']
[[0. 0. 1.]
 [0. 0. 1.]]
unseen category 'X' -> [[0. 0. 0.]]


In [21]:
# The 'dummy variable trap': with all k columns, the columns always sum to 1
d = pd.get_dummies(raw["Embarked"].fillna("S"), dtype=int)
print("row sums:", d.sum(axis=1).unique())

row sums: [1]


## Part F — Scaling

In [22]:
fare = raw[["Fare"]]
scalers = {"Standard": StandardScaler(), "MinMax": MinMaxScaler(), "Robust": RobustScaler()}
rows = {}
for name, sc in scalers.items():
    z = sc.fit_transform(fare).ravel()
    rows[name] = {"min": z.min(), "median": np.median(z), "max": z.max(), "mean": z.mean(), "std": z.std()}
rows["Raw"] = {"min": fare.Fare.min(), "median": fare.Fare.median(), "max": fare.Fare.max(),
               "mean": fare.Fare.mean(), "std": fare.Fare.std(ddof=0)}
print(pd.DataFrame(rows).T.round(3))

            min  median      max    mean     std
Standard -0.648  -0.357    9.667   0.000   1.000
MinMax    0.000   0.028    1.000   0.063   0.097
Robust   -0.626   0.000   21.563   0.769   2.151
Raw       0.000  14.454  512.329  32.204  49.666


In [23]:
mu, sd = raw["Fare"].mean(), raw["Fare"].std(ddof=0)
q1, q3 = raw["Fare"].quantile([0.25, 0.75])
print(f"mean = {mu:.4f}, population std = {sd:.4f}, Q1 = {q1:.4f}, Q3 = {q3:.4f}, IQR = {q3 - q1:.4f}")
x = 100.0
print(f"Fare 100 -> standard: ({x} - {mu:.2f}) / {sd:.2f} = {(x - mu) / sd:.4f}")
print(f"Fare 100 -> min-max : {x} / 512.3292 = {x / 512.3292:.4f}")
print(f"Fare 100 -> robust  : ({x} - {raw['Fare'].median()}) / {q3 - q1:.4f} = {(x - raw['Fare'].median()) / (q3 - q1):.4f}")
print("\nStandardScaler uses ddof=0: pandas .std() (ddof=1) =", round(raw["Fare"].std(), 4))

mean = 32.2042, population std = 49.6655, Q1 = 7.9104, Q3 = 31.0000, IQR = 23.0896
Fare 100 -> standard: (100.0 - 32.20) / 49.67 = 1.3650
Fare 100 -> min-max : 100.0 / 512.3292 = 0.1952
Fare 100 -> robust  : (100.0 - 14.4542) / 23.0896 = 3.7049

StandardScaler uses ddof=0: pandas .std() (ddof=1) = 49.6934


In [24]:
log_fare = np.log1p(raw["Fare"])
print("skewness raw Fare:", round(raw["Fare"].skew(), 3), "| log1p(Fare):", round(log_fare.skew(), 3))

skewness raw Fare: 4.787 | log1p(Fare): 0.395


In [25]:
# Tree models don't care about scaling: same RF accuracy with and without
base_cols = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare"]
b = raw[base_cols + ["Survived"]].copy()
b["Sex"] = (b["Sex"] == "male").astype(int)
b["Age"] = b["Age"].fillna(b["Age"].median())
Xb, yb = b[base_cols], b["Survived"]
Xtr, Xte, ytr, yte = train_test_split(Xb, yb, test_size=0.2, random_state=42)
sc = StandardScaler().fit(Xtr)
for label, (A, B) in {"unscaled": (Xtr, Xte), "scaled": (sc.transform(Xtr), sc.transform(Xte))}.items():
    rf_acc = accuracy_score(yte, RandomForestClassifier(random_state=42).fit(A, ytr).predict(B))
    lr = LogisticRegression(max_iter=5000).fit(A, ytr)
    print(f"{label:9s} RF acc = {rf_acc:.4f} | LR acc = {accuracy_score(yte, lr.predict(B)):.4f} | LR iterations = {lr.n_iter_[0]}")

unscaled  RF acc = 0.7989 | LR acc = 0.8101 | LR iterations = 104


scaled    RF acc = 0.7989 | LR acc = 0.7989 | LR iterations = 8


## Part G — Data leakage and the fix

In [26]:
# Leaky: statistics computed on all rows, then split. Clean: split first, fit on train only.
Xtr_raw, Xte_raw = train_test_split(raw[["Age", "Fare"]], test_size=0.2, random_state=42)
print("Age median all rows :", raw["Age"].median(), "| train rows only:", Xtr_raw["Age"].median())
print("Fare mean  all rows : {:.4f} | train rows only: {:.4f}".format(raw["Fare"].mean(), Xtr_raw["Fare"].mean()))
print("Fare max   all rows : {:.4f} | train rows only: {:.4f}".format(raw["Fare"].max(), Xtr_raw["Fare"].max()))
mm = MinMaxScaler().fit(Xtr_raw[["Fare"]])          # fit on the training rows only
te_scaled = mm.transform(Xte_raw[["Fare"]]).ravel()  # re-use those statistics on the test rows
print("Test fares scaled with the train-fitted scaler: min {:.4f}, max {:.4f}".format(te_scaled.min(), te_scaled.max()))

Age median all rows : 28.0 | train rows only: 28.0
Fare mean  all rows : 32.2042 | train rows only: 32.5863
Fare max   all rows : 512.3292 | train rows only: 512.3292
Test fares scaled with the train-fitted scaler: min 0.0000, max 0.5121


### A leak-proof pipeline
`ColumnTransformer` applies a different recipe to each group of columns; `Pipeline` chains preprocessing and the model, so `fit` learns every statistic from the training rows only and `predict` re-uses them.

In [27]:
def add_features(d):
    d = d.copy()
    d["FamilySize"] = d["SibSp"] + d["Parch"] + 1
    d["IsAlone"] = (d["FamilySize"] == 1).astype(int)
    t = d["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip().replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    d["Title"] = t.where(t.isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")
    d["HasCabin"] = d["Cabin"].notna().astype(int)
    return d

feat = add_features(raw)
num_cols = ["Age", "Fare", "FamilySize"]
cat_cols = ["Sex", "Embarked", "Title"]
pass_cols = ["Pclass", "IsAlone", "HasCabin"]

pre = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_cols),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ("keep", "passthrough", pass_cols),
])
pipe = Pipeline([("prep", pre), ("model", LogisticRegression(max_iter=1000))])

Xf, yf = feat[num_cols + cat_cols + pass_cols], feat["Survived"]
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(Xf, yf, test_size=0.2, random_state=42)
pipe.fit(Xf_tr, yf_tr)
print("Pipeline (engineered features, LR) test accuracy:", round(pipe.score(Xf_te, yf_te), 4))
print("Learned Age median inside the pipeline:", pipe.named_steps["prep"].named_transformers_["num"].named_steps["impute"].statistics_[0])
print("Output feature names:", list(pipe.named_steps["prep"].get_feature_names_out()))

Pipeline (engineered features, LR) test accuracy: 0.8324
Learned Age median inside the pipeline: 28.0
Output feature names: ['num__Age', 'num__Fare', 'num__FamilySize', 'cat__Sex_female', 'cat__Sex_male', 'cat__Embarked_C', 'cat__Embarked_Q', 'cat__Embarked_S', 'cat__Title_Master', 'cat__Title_Miss', 'cat__Title_Mr', 'cat__Title_Mrs', 'cat__Title_Rare', 'keep__Pclass', 'keep__IsAlone', 'keep__HasCabin']


## Part H — Does feature engineering help?
Two feature sets, two models, measured two ways: one 80:20 split (`random_state=42`, as in the manual and the assignment) and **5-fold stratified cross-validation** (more reliable: every row is tested once).

In [28]:
base_num, base_cat, base_pass = ["Age", "Fare", "SibSp", "Parch"], ["Sex", "Embarked"], ["Pclass"]
eng_num, eng_cat, eng_pass = num_cols, cat_cols, pass_cols

def make_pipe(num, cat, passthru, model):
    pre = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat),
        ("keep", "passthrough", passthru),
    ])
    return Pipeline([("prep", pre), ("model", model)])

models = {"LogReg": lambda: LogisticRegression(max_iter=1000),
          "RandomForest": lambda: RandomForestClassifier(n_estimators=100, random_state=42)}
sets = {"baseline": (base_num, base_cat, base_pass), "engineered": (eng_num, eng_cat, eng_pass)}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
for sname, (n, c, p) in sets.items():
    cols = n + c + p
    Xs = feat[cols]
    Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(Xs, yf, test_size=0.2, random_state=42)
    for mname, mk in models.items():
        split_acc = make_pipe(n, c, p, mk()).fit(Xs_tr, ys_tr).score(Xs_te, ys_te)
        cvs = cross_val_score(make_pipe(n, c, p, mk()), Xs, yf, cv=cv)
        results.append({"features": sname, "model": mname, "split acc": split_acc,
                        "CV mean": cvs.mean(), "CV std": cvs.std()})
res = pd.DataFrame(results).round(4)
print(res)

     features         model  split acc  CV mean  CV std
0    baseline        LogReg     0.8101   0.7935  0.0193
1    baseline  RandomForest     0.8045   0.8226  0.0167
2  engineered        LogReg     0.8324   0.8294  0.0094
3  engineered  RandomForest     0.8324   0.8170  0.0168


In [29]:
# Majority-class baseline: always predict 'died'
print("Always predict 0 -> test accuracy:", round((yf_te == 0).mean(), 4), "| overall:", round((yf == 0).mean(), 4))

Always predict 0 -> test accuracy: 0.5866 | overall: 0.6162


In [30]:
# Random-forest importances on the engineered set
rf_pipe = make_pipe(eng_num, eng_cat, eng_pass, RandomForestClassifier(n_estimators=100, random_state=42)).fit(Xf_tr, yf_tr)
names = rf_pipe.named_steps["prep"].get_feature_names_out()
imp = pd.Series(rf_pipe.named_steps["model"].feature_importances_, index=names).sort_values(ascending=False)
print(imp.round(4))

num__Fare            0.2366
num__Age             0.2089
cat__Title_Mr        0.1070
cat__Sex_male        0.0813
cat__Sex_female      0.0809
keep__Pclass         0.0745
num__FamilySize      0.0709
keep__HasCabin       0.0322
cat__Title_Mrs       0.0242
cat__Title_Miss      0.0190
cat__Embarked_S      0.0154
keep__IsAlone        0.0145
cat__Embarked_C      0.0119
cat__Title_Master    0.0100
cat__Embarked_Q      0.0076
cat__Title_Rare      0.0050
dtype: float64


## Part I — Practice-problem solutions

In [31]:
# P1: survival rate of women vs men, and of 1st-class women vs 3rd-class men
r = raw.groupby("Sex")["Survived"].mean().round(4); print(r)
print("1st-class women:", round(raw.query("Sex == 'female' and Pclass == 1")["Survived"].mean(), 4),
      "| 3rd-class men:", round(raw.query("Sex == 'male' and Pclass == 3")["Survived"].mean(), 4))

Sex
female    0.7420
male      0.1889
Name: Survived, dtype: float64


1st-class women: 0.9681 | 3rd-class men: 0.1354


In [32]:
# P2: FamilySize / IsAlone counts
f = raw["SibSp"] + raw["Parch"] + 1
print("FamilySize max:", f.max(), "| alone:", (f == 1).sum(), "| not alone:", (f > 1).sum())

FamilySize max: 11 | alone: 537 | not alone: 354


In [33]:
# P3: min-max and z-score of Age = 40 using the known ages
a = raw["Age"].dropna()
print("min", a.min(), "max", a.max(), "mean", round(a.mean(), 4), "pop std", round(a.std(ddof=0), 4))
print("min-max(40) =", round((40 - a.min()) / (a.max() - a.min()), 4), "| z(40) =", round((40 - a.mean()) / a.std(ddof=0), 4))

min 0.42 max 80.0 mean 29.6991 pop std 14.5163
min-max(40) = 0.4974 | z(40) = 0.7096


In [34]:
# P4: number of columns after get_dummies on Sex (2), Embarked (3), Pclass (3), with and without drop_first
t = raw[["Sex", "Embarked", "Pclass"]].fillna({"Embarked": "S"})
print(pd.get_dummies(t, columns=["Sex", "Embarked", "Pclass"]).shape[1],
      pd.get_dummies(t, columns=["Sex", "Embarked", "Pclass"], drop_first=True).shape[1])

8 5


In [35]:
# P5: AgeGroup counts with the manual's bins (global-median fill) — where did the 177 imputed ages go?
ag_counts = pd.cut(raw["Age"].fillna(raw["Age"].median()), bins=[0, 12, 18, 30, 50, 100],
                   labels=["Child", "Teen", "YoungAdult", "Adult", "Senior"]).value_counts().sort_index()
print(ag_counts)
print("Known-age YoungAdults:", pd.cut(raw["Age"].dropna(), bins=[0, 12, 18, 30, 50, 100]).value_counts().sort_index().iloc[2])

Age
Child          69
Teen           70
YoungAdult    447
Adult         241
Senior         64
Name: count, dtype: int64
Known-age YoungAdults: 270


In [36]:
# P6: Title of the passengers with the rare titles, and their survival
t = raw["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
print(raw.assign(Title=t).loc[~t.isin(["Mr", "Mrs", "Miss", "Master", "Mlle", "Ms", "Mme"])].groupby("Title")["Survived"].agg(["count", "mean"]))

              count      mean
Title                        
Capt              1  0.000000
Col               2  0.500000
Don               1  0.000000
Dr                7  0.428571
Jonkheer          1  0.000000
Lady              1  1.000000
Major             2  0.500000
Rev               6  0.000000
Sir               1  1.000000
the Countess      1  1.000000
